# Experiment 10: Multiple Linear Regression using Gradient Descent

**Objective:** To implement multiple linear regression using optimization via gradient descent.

---

## What is Multiple Linear Regression?

Multiple Linear Regression predicts a target value (y) using **two or more input features** (x1, x2, x3, ...).

The formula is:

```
y = w0 + w1*x1 + w2*x2 + ... + wn*xn
```

- `w0` is the **bias** (intercept)
- `w1, w2, ..., wn` are the **weights** for each feature

## What is Gradient Descent?

Gradient Descent is an **optimization algorithm** that finds the best weights by:
1. Starting with random weights
2. Calculating how wrong the predictions are (the **loss**)
3. Adjusting weights in the direction that reduces the loss
4. Repeating until the loss is small enough

The weight update rule is:
```
w = w - learning_rate * gradient
```

## Step 1: Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

print("Libraries imported successfully!")

---

## Example 1: Predicting House Price (Area + Bedrooms)

We have a small dataset where house price depends on:
- Area (in sq ft)
- Number of bedrooms

We will learn the weights using **Gradient Descent** from scratch.

In [ ]:
# ---- DATASET ----
# Features: [Area (sq ft), Bedrooms]
X = np.array([
    [800,  1],
    [1200, 2],
    [1500, 3],
    [1800, 3],
    [2200, 4],
    [2500, 4],
    [3000, 5]
], dtype=float)

# Target: Price in $1000s
y = np.array([150, 220, 280, 320, 400, 450, 530], dtype=float)

print("Dataset:")
print("Area   Bedrooms   Price($1000s)")
for i in range(len(y)):
    print(f"{int(X[i,0])}    {int(X[i,1])}          {y[i]}")

In [ ]:
# ---- NORMALIZE FEATURES ----
# Normalization makes gradient descent converge faster
# It brings all features to a similar scale (0 to 1)

X_min = X.min(axis=0)
X_max = X.max(axis=0)
X_norm = (X - X_min) / (X_max - X_min)

# Add a column of 1s for the bias term (w0)
m = len(y)                          # number of training examples
X_b = np.c_[np.ones(m), X_norm]    # shape: (7, 3)

print("Normalized features (first 3 rows):")
print(X_b[:3])

In [ ]:
# ---- GRADIENT DESCENT IMPLEMENTATION ----

def compute_loss(X, y, weights):
    """
    Mean Squared Error Loss:
    MSE = (1/2m) * sum( (y_predicted - y_actual)^2 )
    """
    predictions = X.dot(weights)       # y_hat = X * w
    errors = predictions - y           # difference
    loss = (1 / (2 * len(y))) * np.sum(errors ** 2)
    return loss


def gradient_descent(X, y, learning_rate=0.1, iterations=1000):
    """
    Updates weights step by step to minimize loss.
    """
    m = len(y)
    n_features = X.shape[1]
    weights = np.zeros(n_features)     # start with all weights = 0
    loss_history = []

    for i in range(iterations):
        predictions = X.dot(weights)   # current predictions
        errors = predictions - y       # how wrong are we?

        # Gradient = (1/m) * X_transpose * errors
        gradient = (1 / m) * X.T.dot(errors)

        # Update weights
        weights = weights - learning_rate * gradient

        # Save loss every 100 steps
        if i % 100 == 0:
            loss_history.append(compute_loss(X, y, weights))

    return weights, loss_history


# Train the model
weights, loss_history = gradient_descent(X_b, y, learning_rate=0.1, iterations=1000)

print("Learned Weights:")
print(f"  Bias (w0)    : {weights[0]:.4f}")
print(f"  Area (w1)    : {weights[1]:.4f}")
print(f"  Bedrooms (w2): {weights[2]:.4f}")

In [ ]:
# ---- VISUALIZE LOSS CURVE ----
plt.figure(figsize=(8, 4))
plt.plot(range(0, 1000, 100), loss_history, marker='o', color='steelblue')
plt.title("Loss vs. Iterations (Gradient Descent)")
plt.xlabel("Iterations")
plt.ylabel("MSE Loss")
plt.grid(True)
plt.tight_layout()
plt.show()

print("Observation: Loss decreases and becomes stable as weights improve.")

In [ ]:
# ---- COMPARE WITH SKLEARN ----
model = LinearRegression()
model.fit(X_norm, y)

print("\nComparison of Weights:")
print(f"{'Parameter':<15} {'Gradient Descent':>20} {'Sklearn':>15}")
print("-" * 52)
print(f"{'Bias (w0)':<15} {weights[0]:>20.4f} {model.intercept_:>15.4f}")
print(f"{'Area (w1)':<15} {weights[1]:>20.4f} {model.coef_[0]:>15.4f}")
print(f"{'Bedrooms (w2)':<15} {weights[2]:>20.4f} {model.coef_[1]:>15.4f}")

# Predictions
y_pred_gd = X_b.dot(weights)
y_pred_sk = model.predict(X_norm)

print(f"\nR2 Score - Gradient Descent : {r2_score(y, y_pred_gd):.4f}")
print(f"R2 Score - Sklearn          : {r2_score(y, y_pred_sk):.4f}")

---

## Example 2: Predicting Student Marks (Study Hours + Sleep Hours)

In [ ]:
# Features: [Study Hours, Sleep Hours]
X2 = np.array([
    [2, 5],
    [3, 6],
    [4, 7],
    [5, 6],
    [6, 8],
    [7, 7],
    [8, 8],
    [9, 9]
], dtype=float)

# Target: Marks (out of 100)
y2 = np.array([40, 50, 58, 65, 72, 78, 85, 92], dtype=float)

# Normalize
scaler = StandardScaler()
X2_norm = scaler.fit_transform(X2)
m2 = len(y2)
X2_b = np.c_[np.ones(m2), X2_norm]

# Train
weights2, loss2 = gradient_descent(X2_b, y2, learning_rate=0.1, iterations=1000)

print("Learned Weights for Student Marks:")
print(f"  Bias          : {weights2[0]:.4f}")
print(f"  Study Hours   : {weights2[1]:.4f}")
print(f"  Sleep Hours   : {weights2[2]:.4f}")

# Predict for a student who studies 6 hrs and sleeps 7 hrs
new_student = scaler.transform([[6, 7]])
new_student_b = np.c_[np.ones(1), new_student]
predicted_mark = new_student_b.dot(weights2)[0]
print(f"\nPredicted mark for 6 study hrs, 7 sleep hrs: {predicted_mark:.2f}")

---

## Example 3: Predicting Car Price (Age + Mileage + Engine Size)

In [ ]:
# Features: [Age (years), Mileage (1000 km), Engine Size (L)]
X3 = np.array([
    [1,  10, 1.2],
    [2,  25, 1.5],
    [3,  40, 1.5],
    [4,  55, 2.0],
    [5,  70, 1.8],
    [6,  90, 2.0],
    [7, 110, 2.5],
    [8, 130, 1.5]
], dtype=float)

# Target: Price in $1000s
y3 = np.array([18, 15, 13, 11, 10, 8, 7, 5], dtype=float)

# Normalize
scaler3 = StandardScaler()
X3_norm = scaler3.fit_transform(X3)
m3 = len(y3)
X3_b = np.c_[np.ones(m3), X3_norm]

# Train
weights3, loss3 = gradient_descent(X3_b, y3, learning_rate=0.05, iterations=2000)

y3_pred = X3_b.dot(weights3)

print("Car Price Prediction Results:")
print(f"{'Actual':>10} {'Predicted':>12}")
for a, p in zip(y3, y3_pred):
    print(f"${a:>8.1f}k  ${p:>10.2f}k")

print(f"\nR2 Score: {r2_score(y3, y3_pred):.4f}")

---

## Example 4: Effect of Learning Rate on Convergence

A **high learning rate** can overshoot. A **low learning rate** converges slowly. Let's see the difference.

In [ ]:
# Use the house price dataset from Example 1
learning_rates = [0.001, 0.01, 0.1, 0.5]
iterations = 500

plt.figure(figsize=(10, 5))

for lr in learning_rates:
    _, loss_hist = gradient_descent(X_b, y, learning_rate=lr, iterations=iterations)
    steps = list(range(0, iterations, 100))
    plt.plot(steps, loss_hist, marker='o', label=f"LR = {lr}")

plt.title("Effect of Learning Rate on Loss Convergence")
plt.xlabel("Iterations")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print("Observation:")
print("  LR = 0.001  -> Very slow convergence (needs many more iterations)")
print("  LR = 0.01   -> Moderate speed")
print("  LR = 0.1    -> Good balance (converges fast and stable)")
print("  LR = 0.5    -> Might diverge or oscillate for some datasets")

---

## Example 5: Using Sklearn's Built-in Model and Comparing Results

Here we compare our custom gradient descent model with Sklearn's `LinearRegression` on a salary dataset.

In [ ]:
# Dataset: Predict Salary based on Experience, Age, Skill Score
# Features: [Experience (years), Age, Skill Score (0-10)]
X5 = np.array([
    [1, 22, 5],
    [2, 24, 6],
    [3, 26, 6],
    [4, 28, 7],
    [5, 30, 7],
    [6, 32, 8],
    [7, 34, 8],
    [8, 35, 9],
    [9, 37, 9],
    [10, 39, 10]
], dtype=float)

# Target: Salary in $1000s per year
y5 = np.array([30, 35, 40, 48, 55, 62, 70, 78, 85, 95], dtype=float)

# Normalize and train with gradient descent
scaler5 = StandardScaler()
X5_norm = scaler5.fit_transform(X5)
m5 = len(y5)
X5_b = np.c_[np.ones(m5), X5_norm]
weights5, _ = gradient_descent(X5_b, y5, learning_rate=0.1, iterations=2000)

# Sklearn model
sk_model5 = LinearRegression()
sk_model5.fit(X5_norm, y5)

# Predictions
y5_gd = X5_b.dot(weights5)
y5_sk = sk_model5.predict(X5_norm)

# Print comparison table
print(f"{'Actual':>10} {'GD Pred':>12} {'SK Pred':>12}")
print("-" * 36)
for a, g, s in zip(y5, y5_gd, y5_sk):
    print(f"${a:>8.1f}k  ${g:>10.2f}k  ${s:>10.2f}k")

print(f"\nR2 - Gradient Descent : {r2_score(y5, y5_gd):.4f}")
print(f"R2 - Sklearn          : {r2_score(y5, y5_sk):.4f}")

# Visualize predictions
plt.figure(figsize=(8, 4))
plt.plot(range(1, 11), y5, 'o-', label='Actual', color='black')
plt.plot(range(1, 11), y5_gd, 's--', label='Gradient Descent', color='steelblue')
plt.plot(range(1, 11), y5_sk, '^-.', label='Sklearn', color='tomato')
plt.title("Actual vs Predicted Salary")
plt.xlabel("Sample Index")
plt.ylabel("Salary ($1000s)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

---

## Summary: Key Concepts Learned

| Concept | Description |
|---|---|
| Multiple Linear Regression | Predicts target using multiple features |
| Loss Function (MSE) | Measures how wrong the predictions are |
| Gradient | Direction of steepest increase in loss |
| Weight Update | Move weights opposite to gradient |
| Learning Rate | Controls how big each step is |
| Normalization | Scales features to help convergence |
| R2 Score | Measures how well the model fits (1.0 = perfect) |

---

# Exercise Questions

Try solving these problems by modifying the code above.

---

### Question 1
Create a dataset with 3 features: **temperature**, **humidity**, and **wind speed** to predict **ice cream sales**. Use at least 8 data points. Apply gradient descent and report the learned weights and final MSE loss.

---

### Question 2
For the house price dataset in Example 1, run gradient descent with 3 different values of `iterations`: **100, 500, and 2000**. Plot the loss for each and explain which converges best and why.

---

### Question 3
Modify the gradient descent function to also track and return the **weight history** (all weight values at each iteration). Then plot how `w1` (first feature weight) changes over iterations for the student marks dataset in Example 2.

---

### Question 4
Using the salary prediction dataset in Example 5, add a **4th feature** called `certifications` (number of certifications each person has) with values `[0, 0, 1, 1, 2, 2, 3, 3, 4, 4]`. Retrain the model and compare the new R2 score with the original one. Did accuracy improve?

---

### Question 5
Implement a function called `predict_new(X_new, weights, scaler)` that:
1. Takes a new raw input array (not normalized)
2. Applies the same scaler used during training
3. Adds the bias column
4. Returns the predicted value

Test it on the car price dataset (Example 3) to predict the price of a **5-year-old car** with **80,000 km** mileage and **2.0L** engine.

In [ ]:
# Write your answers here

# Question 1



# Question 2



# Question 3



# Question 4



# Question 5

